# Laboratorium 5: Sieci neuronowe w przetwarzaniu tekstu

Klasyfikujemy wydźwięk recenzji z PolEmo 2.0-IN (benchmark KLEJ) w czterech klasach: minus, plus, amb (ambiwalentna) i zero (neutralna). Najpierw klasyczny punkt odniesienia, czyli TF-IDF z regresją logistyczną. Potem własny tokenizer i słownik, a na nich trzy sieci: worek embeddingów (Embedding i GlobalAveragePooling1D), dwukierunkowy LSTM i Conv1D. Wynik najlepszej sieci na zbiorze testowym zanotujcie, w laboratorium 6 porównacie go z HerBERT.

Środowisko: Google Colab z GPU (Środowisko wykonawcze, Zmień typ środowiska wykonawczego, T4 GPU). Keras 3 na backendzie TensorFlow (domyślnym w Colabie). Bez GPU wszystko też zadziała, tylko LSTM trenuje się kilka razy wolniej.

Dane: PolEmo 2.0 (Kocoń i in., 2019) w wersji z benchmarku KLEJ, pliki CSV z Hugging Face (`allegro/klej-polemo2-in`). Źródła podają różne licencje: repozytorium CLARIN-PL (korpus pierwotny) CC BY 4.0, strona KLEJ dla wersji w benchmarku CC BY-NC-SA 4.0, karta na Hugging Face ma tag CC BY-SA 4.0. Do użytku dydaktycznego wystarcza każda z nich; przy innym użyciu przyjmijcie najbardziej restrykcyjną (NC, bez użytku komercyjnego).

In [ ]:
import re
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             confusion_matrix, f1_score)

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", 120)
T_START = time.time()

print("Keras", keras.__version__, "| backend", keras.backend.backend(), "| TensorFlow", tf.__version__)
gpu = tf.config.list_physical_devices("GPU")
print("GPU:", gpu if gpu else "brak, trening na CPU")

Wszystkie parametry, które wpływają na czas i wynik, są w jednej komórce. `MAX_LEN` wybierzecie w zadaniu 3; 256 to wartość startowa.

In [ ]:
MAX_TOKENS = 20000  # rozmiar słownika razem z indeksami 0 (padding) i 1 (OOV)
MAX_LEN = 256       # długość sekwencji po paddingu i obcięciu (zadanie 3)
EMB_DIM = 128       # wymiar embeddingu
LSTM_UNITS = 64     # liczba jednostek LSTM w każdym kierunku
BATCH_SIZE = 64
EPOCHS = 20         # górna granica, EarlyStopping zwykle kończy wcześniej
PATIENCE = 3        # ile epok bez poprawy val_loss czekamy

Gotowa komórka: wczytanie danych z Hugging Face i zamiana etykiet na krótkie nazwy. Kolumna `y` zawiera numer klasy według listy `KLASY`.

In [ ]:
URL_IN = "https://huggingface.co/datasets/allegro/klej-polemo2-in/resolve/main/{}.csv"
train_df = pd.read_csv(URL_IN.format("train"))
valid_df = pd.read_csv(URL_IN.format("valid"))
test_df = pd.read_csv(URL_IN.format("test"))

ETYKIETY = {"__label__meta_minus_m": "minus", "__label__meta_plus_m": "plus",
            "__label__meta_amb": "amb", "__label__meta_zero": "zero"}
KLASY = ["minus", "plus", "amb", "zero"]
OPIS_KLAS = {"minus": "negatywna", "plus": "pozytywna", "amb": "ambiwalentna", "zero": "neutralna"}

for df in (train_df, valid_df, test_df):
    df["klasa"] = df["target"].map(ETYKIETY)
    df["y"] = df["klasa"].map(KLASY.index)
    assert df["klasa"].notna().all()

y_train, y_valid, y_test = (df["y"].to_numpy() for df in (train_df, valid_df, test_df))
print("train", len(train_df), "| valid", len(valid_df), "| test", len(test_df))
train_df.head(3)

Gotowe funkcje pomocnicze. `ocen` liczy accuracy i macro F1 i zapisuje wynik w słowniku `wyniki` (ponowne uruchomienie nadpisuje wiersz, nie dubluje go). `rysuj_historie` i `rysuj_macierz` tylko rysują.

In [ ]:
wyniki = {}


def ocen(nazwa, zbior, y_true, y_pred):
    wiersz = {"model": nazwa, "zbiór": zbior,
              "accuracy": accuracy_score(y_true, y_pred),
              "macro F1": f1_score(y_true, y_pred, average="macro")}
    wyniki[(nazwa, zbior)] = wiersz
    print(f"{nazwa} [{zbior}]: accuracy {wiersz['accuracy']:.4f}, macro F1 {wiersz['macro F1']:.4f}")
    return wiersz


def tabela_wynikow(zbior):
    df = pd.DataFrame([w for w in wyniki.values() if w["zbiór"] == zbior])
    return df.sort_values("macro F1", ascending=False).reset_index(drop=True).round(4)


def przewidz(model, X):
    return model.predict(X, batch_size=256, verbose=0).argmax(axis=1)


def rysuj_historie(historia, tytul):
    h = historia.history
    epoki = np.arange(1, len(h["loss"]) + 1)
    fig, osie = plt.subplots(1, 2, figsize=(11, 3.5))
    for os_, klucz, nazwa in [(osie[0], "loss", "strata"), (osie[1], "accuracy", "accuracy")]:
        os_.plot(epoki, h[klucz], marker="o", label="trening")
        os_.plot(epoki, h["val_" + klucz], marker="o", label="walidacja")
        os_.set_xlabel("epoka")
        os_.set_ylabel(nazwa)
        os_.legend()
    fig.suptitle(tytul)
    plt.tight_layout()
    plt.show()


def rysuj_macierz(y_true, y_pred, tytul, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=range(len(KLASY))),
                           display_labels=KLASY).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_xlabel("klasa przewidziana")
    ax.set_ylabel("klasa prawdziwa")
    ax.set_title(tytul)

## Zadanie 1: Dane

1. Sprawdźcie mapowanie etykiet i policzcie rozkład klas w train, valid i test (liczności i procenty).
2. Narysujcie histogram długości recenzji w słowach (podział po spacjach) i podajcie medianę, 90. i 95. percentyl. Porównajcie medianę długości między klasami.
3. Wypiszcie po dwie recenzje z każdej klasy (pierwsze 300 znaków).
4. Czy zbiór jest niezbalansowany? Dlaczego obok accuracy liczymy macro F1?

In [ ]:
# TODO: tabela liczności klas (KLASY) w train_df, valid_df, test_df i procenty
# Podpowiedź: df["klasa"].value_counts().reindex(KLASY)

In [ ]:
# TODO: długość recenzji w słowach: train_df["sentence"].str.split().str.len()
# describe z percentylami 0.5, 0.9, 0.95, mediana według klasy, histogram z opisanymi osiami

In [ ]:
# TODO: po dwie recenzje z każdej klasy, pierwsze 300 znaków

Odpowiedzi do zadania 1:

1. 
2. 
3. 
4. 

## Zadanie 2: Baseline klasyczny, TF-IDF i regresja logistyczna

1. Zbudujcie `TfidfVectorizer` z unigramami i bigramami (`ngram_range=(1, 2)`), `min_df=2`, `sublinear_tf=True`. Dopasujcie go tylko na train.
2. Wytrenujcie `LogisticRegression` (np. `C=10`, `max_iter=2000`) i oceńcie na walidacji funkcją `ocen`.
3. Wypiszcie po kilka cech (n-gramów) o największych wagach dla każdej klasy.

Dlaczego przy niecałych 6 tys. recenzji TF-IDF jest tak mocnym punktem odniesienia?

In [ ]:
# TODO: tfidf = TfidfVectorizer(...), X_train_tfidf, X_valid_tfidf, X_test_tfidf (fit tylko na train!)
# TODO: logreg = LogisticRegression(C=10, max_iter=2000), fit, ocen("TF-IDF + LR", "valid", y_valid, ...)

In [ ]:
# TODO: 10 n-gramów o największych wagach logreg.coef_[i] dla każdej klasy
# Podpowiedź: tfidf.get_feature_names_out(), np.argsort

Odpowiedzi do zadania 2:



## Zadanie 3: Tokenizacja i słownik

Tokenizację robimy sami, w czystym Pythonie. `keras.layers.TextVectorization` zrobiłaby to samo, ale działa tylko z backendem TensorFlow, a w laboratorium 6 przechodzimy na backend torch.

1. Napiszcie `tokenizuj(tekst)`: małe litery i wyrażenie regularne `[a-ząćęłńóśźż0-9]+`.
2. Zbudujcie słownik z `MAX_TOKENS - 2` najczęstszych tokenów zbioru treningowego. Indeks 0 to padding, 1 to token spoza słownika (OOV), słowa dostają indeksy od 2.
3. Napiszcie `koduj(tekst)`, zwracającą listę indeksów. Policzcie odsetek tokenów OOV na walidacji.
4. Narysujcie histogram długości w tokenach, podajcie percentyle i odsetek recenzji dłuższych niż `MAX_LEN`. Czy 256 to rozsądny wybór?
5. Zamieńcie teksty na macierze `X_train`, `X_valid`, `X_test` funkcją `keras.utils.pad_sequences` z `padding="post"`. Na krótkim przykładzie pokażcie różnicę między `padding="pre"` i `"post"` oraz `truncating="pre"` i `"post"`.

In [ ]:
# TODO: WZORZEC_SLOWA = re.compile(r"[a-ząćęłńóśźż0-9]+"), funkcja tokenizuj(tekst)
# TODO: licznik = Counter(...) na train, slownik {słowo: indeks od 2}, MAX_TOKENS - 2 najczęstszych
# TODO: indeks_do_slowa (odwrotny słownik z 0 -> "<PAD>", 1 -> "<OOV>"), funkcja koduj(tekst)

In [ ]:
# TODO: odsetek tokenów OOV (indeks 1) na walidacji
# TODO: długości w tokenach (train), percentyle, odsetek dłuższych niż MAX_LEN, histogram

In [ ]:
# TODO: do_macierzy(teksty) z keras.utils.pad_sequences(..., maxlen=MAX_LEN, padding="post", truncating="post")
# TODO: X_train, X_valid, X_test; sprawdźcie, czy są wiersze z samym paddingiem
# TODO: przykład [[5, 6, 7], [5, 6, 7, 8, 9, 10, 11]], maxlen=5, wszystkie kombinacje pre/post

Odpowiedzi do zadania 3:



## Zadanie 4: Embedding i GlobalAveragePooling1D

1. Napiszcie funkcję `trenuj(model)`: kompilacja (`adam`, `sparse_categorical_crossentropy`, accuracy), `fit` na `X_train` z walidacją na `X_valid`, `EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)`, `verbose=2`. Niech zwraca historię i wypisuje czas treningu.
2. Zbudujcie model: `Input((MAX_LEN,), dtype="int32")`, `Embedding(MAX_TOKENS, EMB_DIM, mask_zero=True)`, `GlobalAveragePooling1D`, `Dropout(0.3)`, `Dense(4, activation="softmax")`. Ile ma parametrów i gdzie jest ich najwięcej?
3. Wytrenujcie go, narysujcie historię, oceńcie na walidacji (`ocen`).
4. Wytrenujcie ten sam model z `mask_zero=False` i porównajcie.

Co tracimy, uśredniając embeddingi słów? Co dzieje się z paddingiem bez maski?

In [ ]:
# TODO: funkcja trenuj(model): compile, EarlyStopping(monitor="val_loss", patience=PATIENCE,
#       restore_best_weights=True), fit z verbose=2, pomiar czasu, zwraca historię
# TODO: funkcja model_gap(mask_zero=True): Input -> Embedding -> GlobalAveragePooling1D -> Dropout(0.3) -> Dense
modele = {}
# modele["Embedding + GAP"] = model_gap(); modele["Embedding + GAP"].summary()

In [ ]:
# TODO: trening, rysuj_historie, ocen("Embedding + GAP", "valid", ...)

In [ ]:
# TODO: ten sam model z mask_zero=False, trening i ocena ("Embedding + GAP bez maski")

Odpowiedzi do zadania 4:



## Zadanie 5: Bidirectional LSTM i Conv1D

1. BiLSTM: `Embedding(MAX_TOKENS, EMB_DIM, mask_zero=True)`, `Bidirectional(LSTM(LSTM_UNITS))`, `Dropout(0.5)`, `Dense(4, softmax)`. Nie ustawiajcie `recurrent_dropout` ani innej aktywacji niż domyślna: wtedy na GPU Keras użyje szybkiej implementacji cuDNN. Maska z cuDNN wymaga paddingu na końcu (`padding="post"`).
2. Conv1D: `Embedding(MAX_TOKENS, EMB_DIM)` (bez maski), `Conv1D(128, 5, activation="relu")`, `GlobalMaxPooling1D`, `Dropout(0.5)`, `Dense(4, softmax)`.
3. Wytrenujcie oba modele funkcją `trenuj`, narysujcie historie i oceńcie na walidacji. Porównajcie liczbę parametrów, czas epoki i wyniki.

Po co LSTM dwukierunkowy? Co „widzi” filtr Conv1D o szerokości 5 i co robi GlobalMaxPooling1D?

In [ ]:
# TODO: model_bilstm(): Input -> Embedding(mask_zero=True) -> Bidirectional(LSTM(LSTM_UNITS)) -> Dropout(0.5) -> Dense
# TODO: modele["BiLSTM"] = ..., liczba parametrów, trenuj, rysuj_historie, ocen("BiLSTM", "valid", ...)

In [ ]:
# TODO: model_conv1d(): Input -> Embedding -> Conv1D(128, 5, relu) -> GlobalMaxPooling1D -> Dropout(0.5) -> Dense
# TODO: modele["Conv1D"] = ..., liczba parametrów, trenuj, rysuj_historie, ocen("Conv1D", "valid", ...)

Odpowiedzi do zadania 5:



## Zadanie 6: Porównanie i test

1. Wyświetlcie tabelę wyników na walidacji (`tabela_wynikow("valid")`). Wybierzcie najlepszą sieć według macro F1 na walidacji.
2. Dopiero teraz, raz, policzcie wyniki na teście dla TF-IDF + LR i trzech sieci. Po teście niczego już nie zmieniajcie.
3. Dla najlepszej sieci wypiszcie `classification_report` na teście i narysujcie macierz pomyłek (obok macierz dla TF-IDF + LR). Która klasa jest najtrudniejsza i z którą jest mylona?
4. Zanotujcie accuracy i macro F1 najlepszej sieci na teście. Będą potrzebne w laboratorium 6.

In [ ]:
# TODO: tabela_wynikow("valid")

In [ ]:
# TODO: najlepsza_siec = nazwa modelu z najwyższym macro F1 na walidacji (słownik wyniki)
# TODO: pred_test = {nazwa: przewidywania na X_test lub X_test_tfidf}; ocen(nazwa, "test", y_test, pred)
# TODO: tabela z accuracy i macro F1 na valid i test dla czterech modeli

In [ ]:
# TODO: classification_report dla najlepszej sieci na teście, macierze pomyłek (rysuj_macierz) dla niej i TF-IDF + LR

In [ ]:
# TODO: wypiszcie accuracy i macro F1 najlepszej sieci (i TF-IDF + LR) na teście i zanotujcie je na lab 6

Odpowiedzi do zadania 6:



## Pytania kontrolne

Pytania są w instrukcji. Odpowiedzi (po jednym do trzech zdań):

1. 
2. 
3. 
4. 
5. 
6. 

# Zadania rozszerzone

## Zadanie A: Wyuczone embeddingi

1. Wyciągnijcie macierz wag warstwy `Embedding` z modelu Embedding + GAP i z BiLSTM.
2. Napiszcie `sasiedzi(slowo, E, k=8)`: k najbliższych słów według podobieństwa cosinusowego, szukając tylko wśród 5000 najczęstszych słów (indeksy 2 do 5001).
3. Wypiszcie sąsiadów słów „polecam”, „brud”, „lekarz”, „personel”, „drogo” w obu modelach.
4. W modelu Embedding + GAP wynik klasy to (prawie) średnia z iloczynów embeddingów słów i wag warstwy Dense. Policzcie dla każdego słowa „wkład” do każdej klasy (`E @ W + b`) i wypiszcie słowa najbardziej wskazujące na każdą klasę.

Czy sąsiedzi przypominają synonimy? Czego nauczyły się te embeddingi i dlaczego?

In [ ]:
# TODO: macierz_embeddingu(model): wagi warstwy Embedding jako numpy (keras.ops.convert_to_numpy)
# TODO: sasiedzi(slowo, E, k=8): podobieństwo cosinusowe wśród indeksów 2 do 5001
# TODO: sąsiedzi słów "polecam", "brud", "lekarz", "personel", "drogo" w modelu GAP i BiLSTM

In [ ]:
# TODO: wagi W, b ostatniej warstwy Dense modelu GAP; wklad = E_gap[2:5002] @ W + b
# TODO: dla każdej klasy 12 słów o największym wkładzie względem średniej z klas

Odpowiedzi do zadania A:



## Zadanie B: Zmiana domeny, PolEmo 2.0-OUT

Zbiór PolEmo 2.0-OUT ma ten sam zbiór treningowy (medycyna i hotele), ale test pochodzi z innych dziedzin: recenzje produktów i opinie o zajęciach na uczelni.

1. Wczytajcie test OUT (gotowa komórka), porównajcie rozkład klas z testem IN i policzcie odsetek OOV.
2. Oceńcie TF-IDF + LR i najlepszą sieć na teście OUT. Porównajcie z wynikami na teście IN.
3. Klasa zero ma w teście OUT jedną recenzję. Policzcie macro F1 także tylko dla minus, plus i amb (`labels=[0, 1, 2]`) oraz recall każdej klasy w obu domenach.
4. Policzcie, jaką accuracy model miałby na OUT, gdyby recall każdej klasy pozostał taki jak na teście IN, a zmienił się tylko rozkład klas. Narysujcie macierze pomyłek.

Na ile spadek wynika z innego słownictwa, a na ile z innego rozkładu klas?

In [ ]:
URL_OUT = "https://huggingface.co/datasets/allegro/klej-polemo2-out/resolve/main/test.csv"
out_df = pd.read_csv(URL_OUT)
out_df["klasa"] = out_df["target"].map(ETYKIETY)
out_df["y"] = out_df["klasa"].map(KLASY.index)
y_out = out_df["y"].to_numpy()
print("test OUT:", len(out_df), "recenzji")
pd.DataFrame({"test IN": test_df["klasa"].value_counts().reindex(KLASY),
              "test OUT": out_df["klasa"].value_counts().reindex(KLASY, fill_value=0)})

In [ ]:
# TODO: X_out = do_macierzy(...), X_out_tfidf = tfidf.transform(...), odsetek OOV na teście OUT
# TODO: przewidywania TF-IDF + LR i najlepszej sieci na OUT, ocen(nazwa, "test OUT", ...)
# TODO: tabela IN kontra OUT: accuracy, macro F1 (4 klasy), macro F1 dla labels=[0, 1, 2], recall każdej klasy

In [ ]:
# TODO: accuracy, jakiej należałoby się spodziewać na OUT, gdyby recall każdej klasy był taki jak na teście IN,
# a zmienił się tylko rozkład klas: suma(recall_IN[k] * liczność_OUT[k]) / liczba recenzji OUT

In [ ]:
# TODO: macierze pomyłek na teście OUT dla najlepszej sieci i TF-IDF + LR

Odpowiedzi do zadania B:

